# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yasmeenmh90-beep/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import os, sys, subprocess
REPO_URL = "https://github.com/yasmeenmh90-beep/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

%pip -q install duckdb huggingface_hub
import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH = '2026-03'
fact_month = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
print("Connected. Working dir:", os.getcwd())

Connected. Working dir: /content/flyrank-ml-internship-starter


## 1. Two signal checks + my rule

Two signals my rule leans on, checked first against real data.

In [2]:
signal1 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position <= 3  THEN '1-3'
            WHEN gsc_avg_position <= 10 THEN '4-10'
            WHEN gsc_avg_position <= 20 THEN '11-20'
            ELSE '21+'
        END AS position_bucket,
        COUNT(*) AS n,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS avg_ctr
    FROM {fact_month}
    WHERE gsc_impressions > 0
    GROUP BY 1
    ORDER BY MIN(gsc_avg_position)
""").df()
print(signal1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  position_bucket        n   avg_ctr
0             1-3   727362  0.003803
1            4-10  1456122  0.003235
2           11-20   519223  0.003146
3             21+   908354  0.001314


In [3]:
import pandas as pd

feats_vol = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_first_half > 0
""").df()

feats_vol["volume_bucket"] = pd.qcut(feats_vol["imp_first_half"], 4, labels=["Q1_low", "Q2", "Q3", "Q4_high"])
feats_vol["declined"] = (feats_vol["imp_second_half"] < 0.8 * feats_vol["imp_first_half"]).astype(int)
signal2 = feats_vol.groupby("volume_bucket", observed=True).agg(n=("declined", "size"), decline_rate=("declined", "mean"))
print(signal2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                   n  decline_rate
volume_bucket                     
Q1_low         38712      0.426793
Q2             37418      0.307392
Q3             37886      0.291453
Q4_high        37965      0.279389


In [4]:
print("signal1 verdict: MIXED — cliff at 21+, but 1-3/4-10/11-20 are close together")
print(signal1)
print()
print("signal2 verdict: CONFIRMED — clean monotonic decline-rate gradient by volume quartile")
print(signal2)

signal1 verdict: MIXED — cliff at 21+, but 1-3/4-10/11-20 are close together
  position_bucket        n   avg_ctr
0             1-3   727362  0.003803
1            4-10  1456122  0.003235
2           11-20   519223  0.003146
3             21+   908354  0.001314

signal2 verdict: CONFIRMED — clean monotonic decline-rate gradient by volume quartile
                   n  decline_rate
volume_bucket                     
Q1_low         38712      0.426793
Q2             37418      0.307392
Q3             37886      0.291453
Q4_high        37965      0.279389


**Signal 1 — Position vs. CTR: MIXED.** CTR falls off a cliff at position 21+ (0.0013, roughly a third of the top bucket) — very poor positions genuinely suppress clicks. But positions 1-3, 4-10, and 11-20 are far closer together than expected (0.0038 → 0.0032 → 0.0031); the steep, smooth position-1-beats-everything gradient doesn't hold cleanly in the top 20. A rule that separates "21+" from "everything better" captures the real signal; a rule trying to finely rank position 3 vs. 15 would be leaning on a gradient that isn't really there.

**Signal 2 — Volume vs. decline rate: CONFIRMED.** Clean monotonic gradient across volume quartiles (n≈37K–39K each): 42.7% decline rate in the lowest quartile falling to 27.9% in the highest. Low-volume content is meaningfully more likely to decline.

**My rule:** one row = one content item with enough first-half volume to measure (imp_first_half >= 50). Checked in order, first match wins:
1. STEEP_DECLINE — second-half impressions fell more than 30% vs first-half.
2. POOR_POSITION_HIGH_VOLUME — position worse than 20 despite top-quartile first-half volume — informed directly by signal 1's finding that 21+ is where CTR really drops.
3. LOW_ENGAGEMENT — has GA4 data available, and engaged sessions are less than 30% of pageviews.

Everything else gets STABLE and isn't ranked. The action score weights each reason code by severity, so items are ranked by urgency within their reason code.

## 2. Build the ranked queue (writes the CSV)

In [5]:
import numpy as np
import os

feats = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
           SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_second_half,
           AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)       AS pos_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_pageviews ELSE 0 END)   AS pageviews_first_half,
           SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_engaged_sessions ELSE 0 END) AS engaged_sessions_first_half,
           BOOL_OR(ga4_data_available) AS has_ga4
    FROM {fact_month}
    GROUP BY 1, 2
    HAVING imp_first_half >= 50
""").df()
print(len(feats), "content items with enough first-half history")

feats["has_ga4"] = feats["has_ga4"].fillna(False).astype(bool)
feats["pct_change"] = (feats["imp_second_half"] - feats["imp_first_half"]) / feats["imp_first_half"]
vol_p75 = feats["imp_first_half"].quantile(0.75)

def reason_code(row):
    if row["pct_change"] < -0.30:
        return "STEEP_DECLINE"
    if row["pos_first_half"] > 20 and row["imp_first_half"] >= vol_p75:
        return "POOR_POSITION_HIGH_VOLUME"
    if row["has_ga4"] and row["pageviews_first_half"] > 0:
        engagement_rate = row["engaged_sessions_first_half"] / row["pageviews_first_half"]
        if engagement_rate < 0.30:
            return "LOW_ENGAGEMENT"
    return "STABLE"

feats["reason_code"] = feats.apply(reason_code, axis=1)

def action_score(row):
    if row["reason_code"] == "STEEP_DECLINE":
        return abs(row["pct_change"]) * np.log1p(row["imp_first_half"])
    if row["reason_code"] == "POOR_POSITION_HIGH_VOLUME":
        return (row["pos_first_half"] / 100) * np.log1p(row["imp_first_half"])
    if row["reason_code"] == "LOW_ENGAGEMENT":
        eng = row["engaged_sessions_first_half"] / max(row["pageviews_first_half"], 1)
        return (1 - eng) * np.log1p(row["pageviews_first_half"])
    return 0.0

feats["action_score"] = feats.apply(action_score, axis=1)

queue = feats[feats["reason_code"] != "STABLE"].sort_values("action_score", ascending=False)
print("Flagged for review:", len(queue), "of", len(feats))
print(queue["reason_code"].value_counts())

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Written: work/outputs/baseline_action_score.csv")

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

non_circular = queue[queue["reason_code"] != "STEEP_DECLINE"].copy()
non_circular_labels = (non_circular["pct_change"] < -0.20).astype(int)
p_at_20_honest = precision_at_k(non_circular["action_score"].values, non_circular_labels.values, min(20, len(non_circular)))
base_rate_honest = non_circular_labels.mean()
print(f"Base rate (non-circular): {base_rate_honest:.3f}, Precision@20 (non-circular): {p_at_20_honest:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

92548 content items with enough first-half history
Flagged for review: 46442 of 92548
reason_code
LOW_ENGAGEMENT               23098
STEEP_DECLINE                20138
POOR_POSITION_HIGH_VOLUME     3206
Name: count, dtype: int64
Written: work/outputs/baseline_action_score.csv
Base rate (non-circular): 0.088, Precision@20 (non-circular): 0.200


Precision@K note: an earlier version of this check gave a suspicious 1.000 — because STEEP_DECLINE's trigger (pct_change < -0.30) is strictly nested inside the label used to test it (pct_change < -0.20), so any STEEP_DECLINE item auto-passes by construction. The honest test above excludes STEEP_DECLINE and evaluates only POOR_POSITION_HIGH_VOLUME and LOW_ENGAGEMENT, which are genuinely independent of that label.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-10 review

In [7]:
action_label_map = {
    "STEEP_DECLINE": "Investigate for content refresh or consolidation",
    "POOR_POSITION_HIGH_VOLUME": "Prioritize for on-page SEO improvement",
    "LOW_ENGAGEMENT": "Review content-to-intent fit",
}
reason_summary_map = {
    "STEEP_DECLINE": "second-half impressions fell sharply vs first-half",
    "POOR_POSITION_HIGH_VOLUME": "high first-half volume but position worse than 20",
    "LOW_ENGAGEMENT": "GA4 shows engaged sessions under 30% of pageviews",
}
wrong_map = {
    "STEEP_DECLINE": "a real content consolidation or redirect, or a seasonal dip, would look identical to a genuine decline",
    "POOR_POSITION_HIGH_VOLUME": "a very recently published page could still be climbing rankings normally",
    "LOW_ENGAGEMENT": "a quick-answer page (e.g. a converter) is supposed to have short, low-engagement visits — that's success, not failure",
}

feats["action_label"] = feats["reason_code"].map(action_label_map)
queue = feats[feats["reason_code"] != "STABLE"].sort_values("action_score", ascending=False)

top10 = queue.head(10).copy()
top10["why_its_there"] = top10["reason_code"].map(reason_summary_map)
top10["what_would_make_it_wrong"] = top10["reason_code"].map(wrong_map)

print("Reason codes in top 10:")
print(top10["reason_code"].value_counts())
top10[["client_hash_id", "content_hash_id", "action_label", "why_its_there", "what_would_make_it_wrong"]]

Reason codes in top 10:
reason_code
STEEP_DECLINE    10
Name: count, dtype: int64


,client_hash_id,content_hash_id,action_label,why_its_there,what_would_make_it_wrong
88580,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
14956,client_23a62021009f63c4,content_afa44be39cea94ca,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
62440,client_23a62021009f63c4,content_65c75874a23fca87,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
3745,client_65de48885f4ef01b,content_62673eea26c31c17,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
74849,client_62f4a7e64f5e0096,content_6a56a2183dc01691,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
6778,client_62f4a7e64f5e0096,content_945d6ff91386c817,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
86251,client_20259bd6705d81d4,content_f97d175377d97f04,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
46549,client_73cda7b4e4f265ea,content_9bcfb1e373c01b7a,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
10638,client_e5c2aa26a8598242,content_8abf2671c081e29e,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."
6699,client_62f4a7e64f5e0096,content_9e0a8a913953b8d3,Investigate for content refresh or consolidation,second-half impressions fell sharply vs first-...,"a real content consolidation or redirect, or a..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [8]:
weak_picks = top10[top10["imp_first_half"] < 200]
print("Weak picks in top 10 (low volume, noisy %):", len(weak_picks))
print(weak_picks[["content_hash_id", "reason_code", "imp_first_half", "pct_change"]])

score_source = action_score.__code__.co_names
print("\nimp_second_half used in scoring function:", "imp_second_half" in score_source)

cols = con.sql(f"DESCRIBE SELECT * FROM {fact_month} LIMIT 0").df()["column_name"].tolist()
product_flags = [c for c in cols if "score" in c.lower() or "priority" in c.lower() or "health" in c.lower()]
print("Product decision flags present in source table:", product_flags)
print("Month used:", MONTH, "— never the sealed final month or the _sample table")

Weak picks in top 10 (low volume, noisy %): 0
Empty DataFrame
Columns: [content_hash_id, reason_code, imp_first_half, pct_change]
Index: []

imp_second_half used in scoring function: False
Product decision flags present in source table: []
Month used: 2026-03 — never the sealed final month or the _sample table


The naive low-volume weak-pick check on the top 10 found nothing unusual by volume — the real weak pick is structural, visible earlier: the ranked queue's very top is dominated by STEEP_DECLINE, whose raw score (log1p(impressions)) scales up faster than the other two reason codes' formulas, so POOR_POSITION_HIGH_VOLUME and LOW_ENGAGEMENT rarely compete for the top slots even though signal 1 and signal 2 both showed they carry real information. That's the thing I'd fix next: rescale action_score per reason code (e.g. normalize each to its own 0–1 range before ranking) so all three can actually compete for top-10 slots.

imp_second_half is confirmed absent from action_score() — it's used only to compute pct_change for reason-code labeling, never fed into the score itself. No product decision flags exist in this warehouse table, and this notebook used month=2026-03 throughout, never the sealed final month or the _sample table.

## 5. Metrics (committed instead of the CSV)

The CSV regenerates on every run and stays out of git; these numbers are the run's receipts.

In [9]:
import json
metrics = {
    "month": MONTH,
    "signal1_verdict": "MIXED — cliff at position 21+, but 1-3/4-10/11-20 close together",
    "signal2_verdict": "CONFIRMED — clean monotonic decline-rate gradient by volume quartile",
    "signal2_decline_rate_by_quartile": signal2["decline_rate"].to_dict(),
    "n_flagged": len(queue),
    "n_total": len(feats),
    "reason_code_counts": queue["reason_code"].value_counts().to_dict(),
    "reason_code_counts_top10": top10["reason_code"].value_counts().to_dict(),
    "action_labels_used": sorted(top10["action_label"].unique().tolist()),
    "base_rate_non_circular": float(base_rate_honest),
    "precision_at_20_non_circular": float(p_at_20_honest),
}
with open("work/outputs/baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "month": "2026-03",
  "signal1_verdict": "MIXED \u2014 cliff at position 21+, but 1-3/4-10/11-20 close together",
  "signal2_verdict": "CONFIRMED \u2014 clean monotonic decline-rate gradient by volume quartile",
  "signal2_decline_rate_by_quartile": {
    "Q1_low": 0.42679272576978716,
    "Q2": 0.3073921641990486,
    "Q3": 0.29145330729029195,
    "Q4_high": 0.2793889108389306
  },
  "n_flagged": 46442,
  "n_total": 92548,
  "reason_code_counts": {
    "LOW_ENGAGEMENT": 23098,
    "STEEP_DECLINE": 20138,
    "POOR_POSITION_HIGH_VOLUME": 3206
  },
  "reason_code_counts_top10": {
    "STEEP_DECLINE": 10
  },
  "action_labels_used": [
    "Investigate for content refresh or consolidation"
  ],
  "base_rate_non_circular": 0.08823753041362531,
  "precision_at_20_non_circular": 0.2
}


In [10]:
with open(".gitignore") as f:
    content = f.read()
print("work/outputs/*.csv" in content or "*.csv" in content)
print(content)

True
# --- Data leak guard -------------------------------------------------------
# Block every dataset by default. Only the tiny anonymized starter slice ships.
data/**
!data/
!data/raw/
!data/raw/content_refresh_anonymized.csv

# Never commit bulk dataset dumps or archives
*.parquet
*.zip
*.tar
*.tar.gz
*.feather

# --- Generated pipeline artifacts -----------------------------------------
# data/processed is regenerated by the pipeline
data/processed/
# outputs: keep the small committed examples (model_report.md, charts/,
# refresh_queue_sample.csv), ignore regenerated heavy artifacts
outputs/*.pdf
outputs/model_results.json
outputs/summary.json
outputs/*.csv
!outputs/refresh_queue_sample.csv

# --- Intern work area ------------------------------------------------------
# your experiments live in work/; datasets never enter git
work/**/*.csv

# --- Python / notebooks ----------------------------------------------------
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.venv/
venv/
.env



## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.